- Title page
- Table of Contents
- Problem statement and its short explanation of the goal
- Methodology / problem-solving approach
- Key points of your technique/algorithms/ideas for speed-up
- Limitations and challenges encountered
- Test/example functions used, with graphical visualizations and output results
- Computational cost and time-complexity analysis (if you can)
- Ideas for further improvements of the projects
- References

In [307]:
import numpy as np
import torch
from mytorch.tensor.arithmetic import *
from mytorch.tensor.arithmetic import MyConv2d

## Tensor

## Optimization
### Stochastic Gradient Descent
### Adaptive Moment Estimation
### Adaptive Gradient Algorithm
### Optimization Strategy
After having all the optimization algorithms to choose from, there are still three different strategies that you can choose from to update our model.
#### Stochastic Optimization
Stochastic optimization means that we feed the data into the model **one sample at a time** and update the model immediately. The term stochastic indicates the randomness involved in how the updates are computed. This strategy can be formulated as:
$$
\theta \leftarrow \theta - \eta \nabla_\theta l\big(f_\theta(x_i),\, y_i\big)
$$
- $f_\theta$: The model with parameters $\theta$
- $\eta$: The learning rate
- $l$: The loss function
- $x_i$: The input sample of index $i$
- $y_i$: The corresponding target value

However, it has some major disadvantages that make it no longer an ideal strategy:
- One single sample is not representative enough and too noisy, the computed gradients might lead to wrong updates.
- Unstable updates can cause the model to miss the global optimum, which leads to poor convergence.
- Using one sample at a time doesn't make good use of GPU, which leads to slow computation.

#### Batch Optimization
Batch optimization means that the entire dataset is fed into the model and update the model based on the sum or the mean of the losses. This can be formulated as:
$$
\theta \leftarrow \theta - \eta \nabla_\theta \frac{1}{N} \sum_{i=0}^{N - 1} l\big(f_\theta(x_i),\, y_i\big)
$$
- $N$: The size of the entire dataset

Similarly, it also has some disadvantages:
- Training can be extremely slow for very large datasets like [ImageNet](https://www.image-net.org/).
- High memory requirement.
- Tend to stuck in local optimum as no randomness is involved.

#### Mini-batch Optimization
Mini-batch optimization is a trade-off between the above two strategies. We feed a part of the dataset into the model. This can be formulated as:
$$
\theta \leftarrow \theta - \eta \nabla_\theta \frac{1}{B} \sum_{i=0}^{B - 1} l\big(f_\theta(x_i),\, y_i\big)
$$
- $B$: The batch size

It is important to choose a right batch size when using this strategy.

## Linear Model
Linear model is one of the most fundamental machine learning models that is often the building block for many deep neural networks. Its objective is to capture a linear relation between the input samples and target values. Essentially, its forward pass is a [multivariate linear regression](https://en.wikipedia.org/wiki/General_linear_model), given input samples $X$, a weight matrix $W$ and a bias vector $b$, it can be formulated as:
$$
\hat{Y}(i,\, j) = b(j) + X(i,\, 0) W(0,\, j) + X(i,\, 1) W(1,\, j) + \cdots + X(i,\, N_{\text{in}} - 1) W(N_{\text{in}} - 1,\, j) = b(j) + \sum_{k=0}^{N_{\text{in}} - 1} X(i,\, k) W(k,\, j) = (XW + b)(i,\, j)
$$
- $X \in \mathbb{R}^{B \times N_{\text{in}}}$: Input samples, where $B$ is the batch size, $N_{\text{in}}$ is the number of input features
- $W \in \mathbb{R}^{N_{\text{in}} \times N_{\text{out}}}$: Weight matrix, where $N_{\text{out}}$ is the number of output features
- $b \in \mathbb{R}^{N_{\text{out}}}$: Bias vector
- $\hat{Y} \in \mathbb{R}^{B \times N_{\text{out}}}$: Predicted values

In other words, its forward pass is simply a matrix multiplication between the input samples $X$ and the weight matrix $W$, followed by an addition of the bias vector $b$. As for the backward pass, three gradients are required, $\dfrac{\partial L}{\partial X}$, $\dfrac{\partial L}{\partial W}$ and $\dfrac{\partial L}{\partial b}$, where $L$ is the loss function given by $\frac{1}{B} \sum_{i=0}^{B - 1} l\big(\hat{Y}(i),\, Y(i)\big)$ as a criterion for how good the predicted values are comparing with the target values $Y$. Obviously, $\dfrac{\partial L}{\partial b}$ is a vector of ones, so let's start with $\dfrac{\partial L}{\partial X}$, according to the chain rule we have:
\begin{align}
\dfrac{\partial L}{\partial X(p,\, q)} &= \sum_{i=0}^{B - 1} \sum_{j=0}^{N_{\text{out}} - 1} \dfrac{\partial L}{\partial \hat{Y}(i,\, j)} \cdot \dfrac{\partial \hat{Y}(i,\, j)}{\partial X(p,\, q)} \\
&= \sum_{i=0}^{B - 1} \sum_{j=0}^{N_{\text{out}} - 1} \dfrac{\partial L}{\partial \hat{Y}(i,\, j)} \cdot \dfrac{\partial \big(b(j) + \sum_{k=0}^{N_{\text{in}} - 1} X(i,\, k) W(k,\, j)\big)}{\partial X(p,\, q)} \\
&= \sum_{i=0}^{B - 1} \sum_{j=0}^{N_{\text{out}} - 1} \dfrac{\partial L}{\partial \hat{Y}(i,\, j)} \cdot \sum_{k=0}^{N_{\text{in}} - 1} W(k,\, j) \cdot \dfrac{\partial X(i,\, k)}{\partial X(p,\, q)} \\
&= \sum_{i=0}^{B - 1} \sum_{j=0}^{N_{\text{out}} - 1} \dfrac{\partial L}{\partial \hat{Y}(i,\, j)} \cdot \sum_{k=0}^{N_{\text{in}} - 1} W(k,\, j) \cdot \delta(i,\, p) \cdot \delta(k,\, q)\\
&= \sum_{j=0}^{N_{\text{out}} - 1} \dfrac{\partial L}{\partial \hat{Y}(p,\, j)} \cdot W(q,\, j) \\
&= \sum_{j=0}^{N_{\text{out}} - 1} \dfrac{\partial L}{\partial \hat{Y}(p,\, j)} \cdot W^{\top}(j,\, q) \\
\end{align}

Therefore, we have $\dfrac{\partial L}{\partial X} = \dfrac{\partial L}{\partial \hat{Y}} W^{\top}$. Similarly, we also have $\dfrac{\partial L}{\partial W} = X^{\top} \dfrac{\partial L}{\partial \hat{Y}}$.

We run our implementation for the backward pass of the linear model and verify both gradients with `PyTorch`:

In [308]:
torch.manual_seed(42)
X_shape = (6, 4)
W_shape = (4, 2)
X = torch.randint(1, 4, size=X_shape, dtype=torch.float32, requires_grad=True)
X.retain_grad()
W = torch.randint(1, 4, size=W_shape, dtype=torch.float32, requires_grad=True)
W.retain_grad()
pytorch_output = X @ W
mytorch_output = Tensor(X.flatten().tolist(), shape=X_shape, history=History()) @ Tensor(W.flatten().tolist(), shape=W_shape, history=History())
mytorch_output.backward()
pytorch_output.sum().backward()
print('=== ∂L/∂X ===')
print('MyTorch:', mytorch_output.history.parents[0].gradient.tolist())
print('\nPyTorch:', X.grad.tolist())
print('\n=== ∂L/∂W ===')
print('MyTorch:', mytorch_output.history.parents[1].gradient.tolist())
print('\nPyTorch:', W.grad.tolist())

=== ∂L/∂X ===
MyTorch: [[5.0, 4.0, 4.0, 2.0], [5.0, 4.0, 4.0, 2.0], [5.0, 4.0, 4.0, 2.0], [5.0, 4.0, 4.0, 2.0], [5.0, 4.0, 4.0, 2.0], [5.0, 4.0, 4.0, 2.0]]

PyTorch: [[5.0, 4.0, 4.0, 2.0], [5.0, 4.0, 4.0, 2.0], [5.0, 4.0, 4.0, 2.0], [5.0, 4.0, 4.0, 2.0], [5.0, 4.0, 4.0, 2.0], [5.0, 4.0, 4.0, 2.0]]

=== ∂L/∂W ===
MyTorch: [[12.0, 12.0], [15.0, 15.0], [10.0, 10.0], [14.0, 14.0]]

PyTorch: [[12.0, 12.0], [15.0, 15.0], [10.0, 10.0], [14.0, 14.0]]


### 2D Convolution Operation
In functional analysis, [convolution](https://en.wikipedia.org/wiki/Convolution) is a mathematical operation that takes two functions $f(x)$ and $g(x)$, and produces a new function $(f * g)(x)$ by first reflecting one of the functions over the y-axis, then sliding it over the other function, summing the products of their values at each point where they overlap. For example, reflecting $g(x)$ gives $g(-x)$. Then, sliding $g(-x)$ over $f(x)$ by a displacement of $t$ and multiplying their values gives $f(x)g(t - x)$. Finally, summing the products by taking the integral gives us its definition:
$$
(f * g)(t) := \int_{-\infty}^\infty f(\tau) g(t - \tau) \, d\tau.
$$

However, in the field of deep learning, the term convolution is often used to refer to [cross-correlation](https://en.wikipedia.org/wiki/Cross-correlation). For example, CNN actually uses cross-correlation instead of convolution. although the two operations are very similar, the only difference is that cross-correlation does not involve function reflection. For continuous functions $f$ and $g$, cross-correlation is defined as:
$$
(f \star g)(t) := \int_{-\infty}^\infty f(\tau) g(t + \tau) \, d\tau.
$$

For consistency, we will continue to use convolution to refer to the operation used in CNN. Furthermore, since images and kernels can be viewed as discrete functions in 2D, the discrete 2D convolution is defined as:
$$
(I * K)(i, j) := \sum_{p=0}^{H_{\text{k}}-1} \sum_{q=0}^{W_{\text{k}}-1} I(i + p,\, j + q) K(p,\, q).
$$
- $H_{\text{k}}$: The height of the kernel
- $W_{\text{k}}$: The width of the kernel
- $I(i,\, j)$: The value of pixel $(i,\, j)$ in the image
- $K(p,\, q)$: The value of parameter $(p,\, q)$ in the kernel

2D convolution is the core mathematical operation in CNN. The kernel acts as a sliding window over the image, summing the element-wise products between the kernel and the corresponding pixels of the image, like taking a dot product after they are both vectorized. For example, given an image of shape $(5,\, 5)$ and a kernel of shape $(3,\, 3)$:
$$
I =
\begin{bmatrix}
0 & 1 & 1 & 1 & 0 \\
0 & 1 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0 \\
0 & 0 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0
\end{bmatrix},
\quad
K =
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
$$

As you can see, the tiny binary image resembles a handwritten digit $9$, while the kernel is randomly initialized with natural numbers. According to the equation above, we have:

\begin{align}
(I * K)(0,\, 0) &=
\operatorname{vec}\left(
\begin{bmatrix}
0 & 1 & 1 \\
0 & 1 & 0 \\
0 & 1 & 1
\end{bmatrix}
\right)
\cdot
\operatorname{vec}\left(
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
\right) \\
&= \sum_{p=0}^{2} \sum_{q=0}^{2} I(p,\, q) K(p,\, q) \\
&= 27
\end{align}


This is achieved simply by placing the kernel over the top-left corner of the image and calculating the dot product of their vectorized forms. Similarly, we can compute the value of $(I * K)(1,\, 1)$ by sliding the kernel to the right by one column and down by one row, we have:

\begin{align}
(I * K)(1,\, 1) &=
\operatorname{vec}\left(
\begin{bmatrix}
1 & 0 & 1 \\
1 & 1 & 1 \\
0 & 0 & 1
\end{bmatrix}
\right)
\cdot
\operatorname{vec}\left(
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
\right) \\
&= \sum_{p=0}^{2} \sum_{q=0}^{2} I(1 + p,\, 1 + q) K(p,\, q) \\
&= 28
\end{align}


Finally, we can obtain the entire output of shape $(3,\, 3)$ as follows:
$$
I * K =
\begin{bmatrix}
27 & 40 & 23 \\
13 & 28 & 19 \\
22 & 36 & 23
\end{bmatrix}
$$


### Forward Pass
The forward pass performs a 2D convolution. So far, the images we have used are 2D matrices of shape $(H,\, W)$. However, in real-world scenarios, input image is usually a 3D tensors of shape $(C_{\text{in}},\, H,\, W)$, where $C_{\text{in}}$ denotes the number of input channels. For example, a standard RGB image has three channels corresponding to red, green and blue. Therefore, the kernel must also be expanded from $(H_{\text{k}},\, W_{\text{k}})$ to shape $(C_{\text{in}},\, H_{\text{k}},\, W_{\text{k}})$ to match the channel of the input image. One kernel produces one output channel, so in order to generate a multichannel output, multiple kernels are used. In practice, multiple kernels form a 4D tensor of shape $(C_{\text{out}},\, C_{\text{in}},\, H_{\text{k}},\, W_{\text{k}})$, where $C_{\text{out}}$ denotes the number of output channels, and batched input images form a 4D tensor of shape $(B,\, C_{\text{in}},\, H,\, W)$, where $B$ denotes the batch size. The 2D convolution is then computed as:
$$
(I * K)(b,\, c_{\text{out}},\, i,\, j) =
\sum_{c_{\text{in}}=0}^{C_{\text{in}}-1}
\sum_{p=0}^{H_{\text{k}}-1}
\sum_{q=0}^{W_{\text{k}}-1}
I(b,\, c_{\text{in}},\, i + p,\, j + q)\,
K(c_{\text{out}},\, c_{\text{in}},\, p,\, q)
$$

Furthermore, 2D convolution has three main hyperparameters:
- Padding
- Dilation
- stride

Padding determines the number of additional rows and columns added around the original image, usually filled with zeros. It can enable kernels to capture more information near the image boundaries. For a padding of $(P_{\text{h}},\, P_{\text{w}})$ and an image of shape $(H,\, W)$, $P_{\text{h}}$ rows are added to both the top and bottom of the image, and $P_{\text{w}}$ columns are added to both the left and right sides of the image, producing a padded image of shape $(H + 2P_{\text{h}},\, W + 2P_{\text{w}})$. For example, if we apply a padding of $(1,\, 1)$ to the image $I$ shown above, we will get an $I_{\text{padded}}$ of shape $(7,\, 7)$:
$$
I_{\text{padded}} =
\begin{bmatrix}
0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 1 & 1 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 & 1 & 0 & 0 \\
0 & 0 & 1 & 1 & 1 & 0 & 0 \\
0 & 0 & 0 & 0 & 1 & 0 & 0 \\
0 & 0 & 1 & 1 & 1 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0
\end{bmatrix}
$$

Dilation determines the number of zeros inserted between neighboring parameters of the original kernel. It can effectively enlarge the receptive field without increasing the number of parameters. For a dilation of $(D_{\text{h}},\, D_{\text{w}})$ and a kernel of shape $(H_{\text{k}},\, W_{\text{k}})$, $(D_{\text{h}} - 1)$ zeros are inserted between neighboring parameters in each column, and $(D_{\text{w}} - 1)$ zeros are inserted between neighboring parameters in each row, producing a dilated kernel of shape $(D_{\text{h}} H_{\text{k}} - D_{\text{h}} + 1,\, D_{\text{w}} W_{\text{k}} - D_{\text{w}} + 1)$. For example, if we apply a dilation of $(2,\, 2)$ to the kernel $K$ shown above, we will get a $K_{\text{dilated}}$ of shape $(5,\, 5)$:
$$
K_{\text{dilated}} =
\begin{bmatrix}
1 & 0 & 2 & 0 & 3 \\
0 & 0 & 0 & 0 & 0 \\
4 & 0 & 5 & 0 & 6 \\
0 & 0 & 0 & 0 & 0 \\
7 & 0 & 8 & 0 & 9
\end{bmatrix}
$$

Finally, stride determines how the kernel moves over the image. For an arbitrary stride of $(S_{\text{h}}, S_{\text{w}})$, giving padding $(P_{\text{h}}, P_{\text{w}})$ and dilation $(D_{\text{h}}, D_{\text{w}})$, the 2D convolution is then given by the following equation:
$$
(I * K)(b,\, c_{\text{out}},\, i,\, j) =
\sum_{c_{\text{in}}=0}^{C_{\text{in}}-1}
\sum_{p=0}^{H_{\text{k}}-1}
\sum_{q=0}^{W_{\text{k}}-1}
I(b,\, c_{\text{in}},\, i \cdot S_{\text{h}} + p \cdot D_{\text{h}} - P_{\text{h}},\, j \cdot S_{\text{w}} + q \cdot D_{\text{w}} - P_{\text{w}})\,
K(c_{\text{out}},\, c_{\text{in}},\, p,\, q)
$$
- $I(b, c_{\text{in}}, h, w) =
\begin{cases}
I(b, c_{\text{in}}, h, w), & 0 \le h < H,\, 0 \le w < W \\
0, & \text{otherwise}
\end{cases}$

The shape of the output is given by:
$$
(B,\, C_{\text{out}},\, H_{\text{out}},\, W_{\text{out}})
$$
- $H_{\text{out}} = \left\lfloor \dfrac{H + 2 P_\text{h} - D_\text{h} (H_\text{k} - 1) - 1}{S_\text{h}} + 1 \right\rfloor$
- $W_{\text{out}} = \left\lfloor \dfrac{W + 2 P_\text{w} - D_\text{w} (W_\text{k} - 1) - 1}{S_\text{w}} + 1 \right\rfloor$

This is the equation we use to compute the 2D convolution in practice. Next, we will discuss some different algorithms to implement the 2D convolution, as well as their pros and cons.

#### Nested for-loop
The most straightforward way to compute the 2D convolution is to use a nested for-loop. In short, this algorithm strictly follows the math equation by iterating over every batch, channel, pixel and parameter. The corresponding pseudocode is shown below:
```
for b in range(B):
    for c_out in range(C_out):
        for i in range(H_out):
            for j in range(W_out):
                sum = 0
                for c_in in range(C_in):
                    for p in range(H_k):
                        for q in range(W_k):
                            sum += I_padded[b, c_in, i * S_h + p, j * S_w + q] * K_dilated[c_out, c_in, p, q]
                output[b, c_out, i, j] = s
```
Pros:
- It is conceptually simple, clearly reflects the definition of 2D convolution, easy to understand and implement.
- It has a clear structure, can be easily modified for debugging, visualization or other purpose.

Cons:
- It is computationally expensive, especially for large input images or deep networks.
- It is extremely slow due to the seven for-loops, impractical in real life.
- Its structure can't take advantage of parallel computing using CPU or GPU, slower than other optimized algorithm with vectorized structure.
- Its time complexity given by $\mathcal{O}\big(B \cdot C_{\text{out}} \cdot C_{\text{in}} \cdot H_{\text{out}} \cdot W_{\text{out}} \cdot H_{\text{k}} \cdot W_{\text{k}}\big)$ grows rapidly with these hyperparameters.

#### Im2Col
Im2Col stands for image to column, is a widely used technique for speeding up 2D convolution by converting it into a **matrix multiplication**. As the name suggests, its core idea is to flatten the part of the input image that overlaps the kernel into a column, and kernels are flattened into rows. Then, 2D convolution is computed as a single matrix multiplication which can be highly optimized on modern hardware. For example, consider an input image $I$ of shape $(H = 5, W = 5)$ and a kernel $K$ of shape $(H_{\text{k}} = 3,\, W_{\text{k}} = 3)$:
$$
I =
\begin{bmatrix}
0 & 1 & 1 & 1 & 0 \\
0 & 1 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0 \\
0 & 0 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0
\end{bmatrix},
\quad
K =
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
$$
Sliding $K$ over $I$ with a stride of $(S_{\text{h}} = 2, S_{\text{w}} = 2)$ gives $4$ parts of the image $I$:
$$
I_1 =
\begin{bmatrix}
0 & 1 & 1 \\
0 & 1 & 0 \\
0 & 1 & 1
\end{bmatrix},
\quad
I_2 =
\begin{bmatrix}
1 & 1 & 0 \\
0 & 1 & 0 \\
1 & 1 & 0
\end{bmatrix},
\quad
I_3 =
\begin{bmatrix}
0 & 1 & 1 \\
0 & 0 & 0 \\
0 & 1 & 1
\end{bmatrix},
\quad
I_4 =
\begin{bmatrix}
1 & 1 & 0 \\
0 & 1 & 0 \\
1 & 1 & 0
\end{bmatrix}
$$
Then the $4$ matrices of shape $(3,\, 3)$ are flattened into $4$ column vectors of length $9$ and are stacked into a new matrix of shape $(9,\, 4)$. The kernel $K$ is flattened into a row vector of length $9$:
$$
K_{\text{new}} =
\begin{bmatrix}
1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 9
\end{bmatrix},
\quad
I_{\text{new}} =
\begin{bmatrix}
0 & 1 & 0 & 1 \\
1 & 1 & 1 & 1 \\
1 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 \\
1 & 1 & 0 & 1 \\
0 & 0 & 0 & 0 \\
0 & 1 & 0 & 1 \\
1 & 1 & 1 & 1 \\
1 & 0 & 1 & 0
\end{bmatrix}
$$
Matrix multiplication between $K_{\text{new}}$ and $I_{\text{new}}$ gives a row vector of length $4$:
$$
K_{\text{new}}I_{\text{new}} =
\begin{bmatrix}
27 & 23 & 22 & 23
\end{bmatrix}
$$
Finally, since the expected output shape is $\left(\dfrac{H - H_{\text{k}}}{S_{\text{h}}} + 1 = 2,\, \dfrac{W - W_{\text{k}}}{S_{\text{w}}} + 1 = 2 \right)$, the row vector needs to be reshaped back to $(2,\, 2)$:
$$
\text{output} =
\begin{bmatrix}
27 & 23 \\
22 & 23
\end{bmatrix}
$$
This is a simple example with 2D input. In practice, consider batched input images $I$ of shape $(B,\, C_{\text{in}},\, H,\, W)$ and kernels $K$ of shape $(C_{\text{out}},\, C_{\text{in}},\, H_{\text{k}},\, W_{\text{k}})$, the output shape is then given by $(B,\, C_{\text{out}},\, H_{\text{out}},\, W_{\text{out}})$. First, $I$ is converted to $I_{\text{new}}$ of shape $(B,\, H_{\text{out}},\, W_{\text{out}},\, C_{\text{in}} \cdot H_{\text{k}} \cdot W_{\text{k}})$ and $K$ is converted to $K_{\text{new}}$ of shape $(C_{\text{out}},\, C_{\text{in}} \cdot H_{\text{k}} \cdot W_{\text{k}})$. Then, matrix multiplication $I_{\text{new}} K_{\text{new}}^{\mathsf{T}}$ gives a new tensor of shape $(B,\, H_{\text{out}},\, W_{\text{out}},\, C_{\text{out}})$. Finally, the new tensor needs to be permuted back to $(B,\, C_{\text{out}},\, H_{\text{out}},\, W_{\text{out}})$ to produce the output.

The Im2Col algorithm is used in our custom implementation of 2D convolution. To demonstrate our implementation, consider the following example:
- $I \in \mathbb{R}^{B \times C_{\text{in}} \times H \times W} = \mathbb{R}^{4 \times 3 \times 10 \times 10}$
- $K \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} \times H_{\text{k}} \times W_{\text{k}}} = \mathbb{R}^{2 \times 3 \times 5 \times 5}$
- Stride: $(S_{\text{h}} = 2,\, S_{\text{w}} = 2)$
- Padding: $(P_{\text{h}} = 1,\, P_{\text{w}} = 1)$
- Dilation: $(D_{\text{h}} = 2,\, D_{\text{w}} = 2)$

We run our implementation and compare its output with the result obtained with `PyTorch`:

In [309]:
torch.manual_seed(42)
I_shape = (2, 2, 3, 3)
K_shape = (2, 2, 2, 2)
stride = (2, 2)
padding = (1, 1)
dilation = (2, 2)
I = torch.randint(1, 4, size=I_shape, dtype=torch.float32, requires_grad=True)
I.retain_grad()
K = torch.randint(1, 4, size=K_shape, dtype=torch.float32, requires_grad=True)
K.retain_grad()
pytorch_output = torch.nn.functional.conv2d(I, K, stride=stride, padding=padding, dilation=dilation)
mytorch_output = MyConv2d.forward(Tensor(I.flatten().tolist(), shape=I_shape, history=History()), 
                           Tensor(K.flatten().tolist(), shape=K_shape, history=History()), stride=stride, padding=padding, dilation=dilation)
print("=== 2D Convolution Output ===")
print('MyTorch:', mytorch_output.tolist())
print('\nPyTorch:', pytorch_output.tolist())


=== 2D Convolution Output ===
MyTorch: [[[[2.0, 4.0], [4.0, 6.0]], [[3.0, 4.0], [6.0, 2.0]]], [[[3.0, 6.0], [7.0, 9.0]], [[5.0, 7.0], [9.0, 3.0]]]]

PyTorch: [[[[2.0, 4.0], [4.0, 6.0]], [[3.0, 4.0], [6.0, 2.0]]], [[[3.0, 6.0], [7.0, 9.0]], [[5.0, 7.0], [9.0, 3.0]]]]


### Backward Pass
Most students who have studied 2D convolution know how its forward pass works. However, few of them understand the mechanism of its backward pass during backpropagation. In order to train a CNN, given a scalar-valued differentiable loss function $L$, two gradients are required, one is the gradient of $L$ with respect to the kernel parameters and the other one is the gradient of $L$ with respect to the input image pixels. The key takeaway from this section is that **the backward pass of 2D convolution also uses 2D convolution**. Let's start with the kernel parameters, recall the equation of 2D convolution:
$$
(I * K)(b,\, c_{\text{out}},\, i,\, j) =
\sum_{c_{\text{in}}=0}^{C_{\text{in}}-1}
\sum_{p=0}^{H_{\text{k}}-1}
\sum_{q=0}^{W_{\text{k}}-1}
I(b,\, c_{\text{in}},\, i \cdot S_{\text{h}} + p \cdot D_{\text{h}} - P_{\text{h}},\, j \cdot S_{\text{w}} + q \cdot D_{\text{w}} - P_{\text{w}})\,
K(c_{\text{out}},\, c_{\text{in}},\, p,\, q)
$$

Now that we want to compute $\dfrac{\partial L}{\partial K(c_{\text{out}},\, c_{\text{in}},\, p,\, q)}$, according to the chain rule, we have:


\begin{align}
\dfrac{\partial L}{\partial K(c_{\text{out}},\, c_{\text{in}},\, p,\, q)} &= \sum_{b=0}^{B - 1} \sum_{i=0}^{H_{\text{out}} - 1} \sum_{j=0}^{W_{\text{out} - 1}} \dfrac{\partial L}{\partial (I * K)(b,\, c_{\text{out}},\, i,\, j)} \cdot \dfrac{\partial (I * K)(b,\, c_{\text{out}},\, i,\, j)}{\partial K(c_{\text{out}},\, c_{\text{in}},\, p,\, q)} \\
&= \sum_{b=0}^{B - 1} \sum_{i=0}^{H_{\text{out}} - 1} \sum_{j=0}^{W_{\text{out} - 1}} \dfrac{\partial L}{\partial (I * K)(b,\, c_{\text{out}},\, i,\, j)} \cdot I(b,\, c_{\text{in}},\, i \cdot S_{\text{h}} + p \cdot D_{\text{h}} - P_{\text{h}},\, j \cdot S_{\text{w}} + q \cdot D_{\text{w}} - P_{\text{w}}) \\
&= \sum_{b=0}^{B - 1} \sum_{i=0}^{H_{\text{out}} - 1} \sum_{j=0}^{W_{\text{out} - 1}} I(b,\, c_{\text{in}},\, p \cdot D_{\text{h}} + i \cdot S_{\text{h}} - P_{\text{h}},\, q \cdot D_{\text{w}} + j \cdot S_{\text{w}} - P_{\text{w}}) \cdot G(b,\, c_{\text{out}},\, i,\, j)
\end{align}

- $G(b,\, c_{\text{out}},\, i,\, j)$: The upstream gradient

As you can see, this equation already shows some similarities with a 2D convolution. We just need to permute the first two dimensions of tensors $I$ and $G$, $\dfrac{\partial L}{\partial K}$ is permuted accordingly:
$$
\dfrac{\partial L}{\partial K(c_{\text{in}},\, c_{\text{out}},\, p,\, q)} = \sum_{b=0}^{B - 1} \sum_{i=0}^{H_{\text{out}} - 1} \sum_{j=0}^{W_{\text{out} - 1}} I(c_{\text{in}},\, b,\, p \cdot D_{\text{h}} + i \cdot S_{\text{h}} - P_{\text{h}},\, q \cdot D_{\text{w}} + j \cdot S_{\text{w}} - P_{\text{w}}) \cdot G(c_{\text{out}},\, b,\, i,\, j)
$$

Now this equation becomes a 2D convolution of the input images $I$ with the upstream gradient $G$ and its hyperparameters are as follows:
- Stride: $(D_{\text{h}},\, D_{\text{w}})$
- Padding: $(P_{\text{h}},\, P_{\text{w}})$
- Dilation: $(S_{\text{h}},\, S_{\text{w}})$
- Batch size: $C_{\text{in}}$
- Output channel: $C_{\text{out}}$
- Input channel: $B$

It is interesting to see that the dilation of the forward-pass 2D convolution actually becomes the stride of the backward-pass 2D convolution and how the other hyperparameters are related. Finally, in order to match the shape of the kernels $K$, we need to permute $\dfrac{\partial L}{\partial K(c_{\text{in}},\, c_{\text{out}},\, p,\, q)}$ back to $\dfrac{\partial L}{\partial K(c_{\text{out}},\, c_{\text{in}},\, p,\, q)}$. Formulating the backward pass as a 2D convolution allows us to utilize the highly optimized 2D convolution, instead of computing the gradients separately using some other algorithms.

Similarly, let's continue with the gradient of $L$ with respect to the input image pixels. First, we rewrite the equation of 2D convolution in terms of the dilated kernels $K_{\text{d}}$, so that the dilation $(D_{\text{h}},\, D_{\text{w}})$ becomes $(1,\, 1)$:
$$
(I * K_{\text{d}})(b,\, c_{\text{out}},\, i,\, j) =
\sum_{c_{\text{in}}=0}^{C_{\text{in}}-1}
\sum_{p=0}^{H_{\text{k}_\text{d}}-1}
\sum_{q=0}^{W_{\text{k}_\text{d}}-1}
I(b,\, c_{\text{in}},\, i \cdot S_{\text{h}} + p - P_{\text{h}},\, j \cdot S_{\text{w}} + q - P_{\text{w}})\,
K_{\text{d}}(c_{\text{out}},\, c_{\text{in}},\, p,\, q)
$$

Since a pixel of the input images is denoted by $I(b,\, c_{\text{in}},\, h,\, w)$, we have $h = i \cdot S_{\text{h}} + p - P_{\text{h}}$ and $w = j \cdot S_{\text{w}} + q - P_{\text{w}}$. Then the gradient can be computed as follows:

\begin{align}
\dfrac{\partial L}{\partial I(b,\, c_{\text{in}},\, h,\, w)} &= \sum_{c_{\text{out}}=0}^{C_{\text{out}} - 1} \sum_{i=0}^{H_{\text{out}} - 1} \sum_{j=0}^{W_{\text{out} - 1}} \dfrac{\partial L}{\partial (I * K_{\text{d}})(b,\, c_{\text{out}},\, i,\, j)} \cdot \dfrac{\partial (I * K_{\text{d}})(b,\, c_{\text{out}},\, i,\, j)}{\partial I(b,\, c_{\text{in}},\, h,\, w)} \\
&= \sum_{c_{\text{out}}=0}^{C_{\text{out}} - 1} \sum_{i=0}^{H_{\text{out}} - 1} \sum_{j=0}^{W_{\text{out} - 1}} \dfrac{\partial L}{\partial (I * K_{\text{d}})(b,\, c_{\text{out}},\, i,\, j)} \cdot K_{\text{d}}(c_{\text{out}},\, c_{\text{in}},\, p,\, q) \\
&= \sum_{c_{\text{out}}=0}^{C_{\text{out}} - 1} \sum_{i=0}^{H_{\text{out}} - 1} \sum_{j=0}^{W_{\text{out} - 1}} K_{\text{d}}(c_{\text{out}},\, c_{\text{in}},\, h - i \cdot S_{\text{h}} + P_{\text{h}},\, w - j \cdot S_{\text{w}} + P_{\text{w}}) \cdot G(b,\, c_{\text{out}},\, i,\, j)
\end{align}


We also need to permute the first two dimensions of tensors $K_{\text{d}}$, $\dfrac{\partial L}{\partial I}$ is permuted accordingly:
$$
\dfrac{\partial L}{\partial I(c_{\text{in}},\, b,\, h,\, w)} = \sum_{c_{\text{out}}=0}^{C_{\text{out}} - 1} \sum_{i=0}^{H_{\text{out}} - 1} \sum_{j=0}^{W_{\text{out} - 1}} K_{\text{d}}(c_{\text{in}},\, c_{\text{out}},\, h - i \cdot S_{\text{h}} + P_{\text{h}},\, w - j \cdot S_{\text{w}} + P_{\text{w}}) \cdot G(b,\, c_{\text{out}},\, i,\, j)
$$

According to the equation of 2D convolution, we would expect to get terms like $h + i$ and $w + j$, but instead we get $h - i$ and $w - j$. Therefore, an important trick hare is to rotate the upstream gradient $G$ by $180^{\circ}$ along its last two dimensions, denoted by $G_{\text{r}}$. An example of rotating a matrix by $180^{\circ}$ is as follows:

$$
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
\rightarrow
\begin{bmatrix}
9 & 8 & 7 \\
6 & 5 & 4 \\
3 & 2 & 1
\end{bmatrix}
$$
Therefore, after the rotation we have:

\begin{align}
\dfrac{\partial L}{\partial I(c_{\text{in}},\, b,\, h,\, w)} &= \sum_{c_{\text{out}}=0}^{C_{\text{out}} - 1} \sum_{i=0}^{H_{\text{out}} - 1} \sum_{j=0}^{W_{\text{out} - 1}} K_{\text{d}}(c_{\text{in}},\, c_{\text{out}},\, h - i \cdot S_{\text{h}} + P_{\text{h}},\, w - j \cdot S_{\text{w}} + P_{\text{w}}) \cdot G_{\text{r}}(b,\, c_{\text{out}},\, (H_{\text{out}} - 1) - i,\, (W_{\text{out}} - 1) - j) \\
&= \sum_{c_{\text{out}}=0}^{C_{\text{out}} - 1} \sum_{i=0}^{H_{\text{out}} - 1} \sum_{j=0}^{W_{\text{out} - 1}} K_{\text{d}} \Bigg(c_{\text{in}},\, c_{\text{out}},\, h + (H_{\text{out}} - 1 - i) \cdot S_{\text{h}} - \bigg((H_{\text{out}} - 1) \cdot S_{\text{h}} - P_{\text{h}}\bigg) \\
&\quad ,\, w + (W_{\text{out}} - 1 - j) \cdot S_{\text{w}} - \bigg((W_{\text{out}} - 1) \cdot S_{\text{w}} - P_{\text{w}}\bigg)\Bigg) \cdot G_{\text{r}}\Bigg(b,\, c_{\text{out}},\, H_{\text{out}} - 1 - i,\, W_{\text{out}} - 1 - j\Bigg) \\
&= \sum_{c_{\text{out}}=0}^{C_{\text{out}} - 1} \sum_{i'=0}^{H_{\text{out}} - 1} \sum_{j'=0}^{W_{\text{out} - 1}} K_{\text{d}} \Bigg(c_{\text{in}},\, c_{\text{out}},\, h + i' \cdot S_{\text{h}} - \bigg((H_{\text{out}} - 1) \cdot S_{\text{h}} - P_{\text{h}}\bigg),\, w + j' \cdot S_{\text{w}} - \bigg((W_{\text{out}} - 1) \cdot S_{\text{w}} - P_{\text{w}}\bigg)\Bigg) \cdot G_{\text{r}}\Bigg(b,\, c_{\text{out}},\, i',\, j'\Bigg)
\end{align}

- $i' = H_{\text{out}} - 1 - i$
- $j' = W_{\text{out}} - 1 - j$

This equation gives us the gradient of $L$ with respect to the input image pixels. It is a 2D convolution of the dilated kernels $K_{\text{d}}$ with the rotated upstream gradient $G_{\text{r}}$ and its hyperparameters are as follows:
- Stride: $(1,\, 1)$
- Padding: $\bigg((H_{\text{out}} - 1) \cdot S_{\text{h}} - P_{\text{h}},\, (W_{\text{out}} - 1) \cdot S_{\text{w}} - P_{\text{w}}\bigg)$
- Dilation: $(S_{\text{h}},\, S_{\text{w}})$
- Batch size: $C_{\text{in}}$
- Output channel: $B$
- Input channel: $C_{\text{out}}$

Finally, in order to match the shape of the input images $I$, we need to permute $\dfrac{\partial L}{\partial I(c_{\text{in}},\, b,\, h,\, w)}$ back to $\dfrac{\partial L}{\partial I(b,\, c_{\text{in}},\, h,\, w)}$.

Our implementation of the backward pass strictly follows the two equations of $\dfrac{\partial L}{\partial K}$ and $\dfrac{\partial L}{\partial I}$. Again, we use the same example as in the forward pass, run our backward pass and verify both gradients with `PyTorch`:

In [310]:
mytorch_output.backward()
pytorch_output.sum().backward()
print('=== ∂L/∂I ===')
print('MyTorch:', mytorch_output.history.parents[0].gradient.tolist())
print('\nPyTorch:', I.grad.tolist())
print('\n=== ∂L/∂K ===')
print('MyTorch:', mytorch_output.history.parents[1].gradient.tolist())
print('\nPyTorch:', K.grad.tolist())

=== ∂L/∂I ===
MyTorch: [[[[0.0, 0.0, 0.0], [0.0, 18.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 13.0, 0.0], [0.0, 0.0, 0.0]]], [[[0.0, 0.0, 0.0], [0.0, 18.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 13.0, 0.0], [0.0, 0.0, 0.0]]]]

PyTorch: [[[[0.0, 0.0, 0.0], [0.0, 18.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 13.0, 0.0], [0.0, 0.0, 0.0]]], [[[0.0, 0.0, 0.0], [0.0, 18.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 13.0, 0.0], [0.0, 0.0, 0.0]]]]

=== ∂L/∂K ===
MyTorch: [[[[3.0, 3.0], [3.0, 3.0]], [[2.0, 2.0], [2.0, 2.0]]], [[[3.0, 3.0], [3.0, 3.0]], [[2.0, 2.0], [2.0, 2.0]]]]

PyTorch: [[[[3.0, 3.0], [3.0, 3.0]], [[2.0, 2.0], [2.0, 2.0]]], [[[3.0, 3.0], [3.0, 3.0]], [[2.0, 2.0], [2.0, 2.0]]]]


### Training


### References

- deepkumarpatra. (23 Jul, 2025). Math behind Convolutional Neural Networks. https://www.geeksforgeeks.org/deep-learning/math-behind-convolutional-neural-networks/
- Jefkine. (5 Sep, 2016). Backpropagation in Convolutional Neural Networks. https://www.jefkine.com/general/2016/09/05/backpropagation-in-convolutional-neural-networks/
- alka1974. (30 Sep, 2025). Mini-Batch Gradient Descent in Deep Learning. https://www.geeksforgeeks.org/deep-learning/mini-batch-gradient-descent-in-deep-learning/
- rahulsm27. (23 Jul, 2025). Kaiming Initialization in Deep Learning. https://www.geeksforgeeks.org/deep-learning/kaiming-initialization-in-deep-learning/
- Thomas Kurbiel. (Apr 22, 2021). Derivative of the Softmax Function and the Categorical Cross-Entropy Loss. https://medium.com/data-science/derivative-of-the-softmax-function-and-the-categorical-cross-entropy-loss-ffceefc081d1
- Josh Levy-Kramer. (Jan 15, 2025). Gradients of Matrix Multiplication in Deep Learning. https://robotchinwag.com/posts/gradient-of-matrix-multiplicationin-deep-learning/